# **TFM: Detecció d'esdeveniments importants en partits de futbol a partir de les seves narracions**

**Autor:** Martí Mullor Rordíguez

**Institució:** Universitat Oberta de Catalunya  
**Tutor:** Josep Mª Carmona Leyva

**Data:** Juliol 2026

---

### **Nom de l'script: 03_RoBERTa**

Aquest script entrena un model Transformer (RoBERTa) per a la classificació de text.

Està compost per:

**0. Importacions**

**1. Configuració inicial i càrrega de dades**

1.1 Muntar Google Drive

1.2 Carregar la configuració inicial

1.3 Definició de rutes i extracció de paràmetres i dades

**2. Preprocessament de text**

**3. Entrenament del model RoBERTa**

**4. Avaluació i exportació de resultats**

---


## **0. Importacions**

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from google.colab import drive
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from sklearn.utils.class_weight import compute_class_weight

## **1. Configuració inicial i càrrega de dades**

### **1.1. Muntar Google Drive**

In [ ]:
if not os.path.exists('/content/drive'):
    print("Muntant Google Drive...")
    drive.mount('/content/drive')

### **1.2. Carregar la configuració inicial**

In [ ]:
CONFIG_PATH = "/content/drive/MyDrive/TFM/TFM-Deteccio-Esdeveniments-Futbol/config.json"
with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

### **1.3. Definició de rutes i extracció de paràmetres i dades**

In [ ]:
paths = config["paths"]
seed = config["global_settings"]["random_seed"]
roberta_params = config["roberta"]

# Aquí fixo la llavor:
torch.manual_seed(seed)
np.random.seed(seed)

# Carregar dades
print("Carregant els conjunts de dades preprocessats...")
df_train = pd.read_pickle(os.path.join(paths["processed_data"], "train_dataset.pkl"))
df_val = pd.read_pickle(os.path.join(paths["processed_data"], "val_dataset.pkl"))
df_test = pd.read_pickle(os.path.join(paths["processed_data"], "test_dataset.pkl"))

print(f"Mida Train: {df_train.shape} | Val: {df_val.shape} | Test: {df_test.shape}")

**2. Preprocessament de text**

In [ ]:
print("Codificant les etiquetes (Label Encoding)...")
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(df_train['label'])
y_val_encoded = label_encoder.transform(df_val['label'])
y_test_encoded = label_encoder.transform(df_test['label'])

num_labels = len(label_encoder.classes_)
print(f"Nombre de classes detectades: {num_labels}")

print("Tokenitzant el text amb RobertaTokenizer...")
tokenizer = RobertaTokenizer.from_pretrained(roberta_params["model_name"])

def tokenize_data(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=roberta_params["max_length"],
        return_tensors="pt"
    )

train_encodings = tokenize_data(df_train['text'])
val_encodings = tokenize_data(df_val['text'])
test_encodings = tokenize_data(df_test['text'])

# Creació d'un Dataset per a PyTorch
class SoccerDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = SoccerDataset(train_encodings, y_train_encoded)
val_dataset = SoccerDataset(val_encodings, y_val_encoded)
test_dataset = SoccerDataset(test_encodings, y_test_encoded)

**3. Enrenament del model RoBERTa**

In [ ]:
print("Inicialitzant el model RoBERTa...")
model = RobertaForSequenceClassification.from_pretrained(
    roberta_params["model_name"],
    num_labels=num_labels
)

training_args = TrainingArguments(
    output_dir=os.path.join(paths["models"], "roberta_checkpoints"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=roberta_params["learning_rate"],
    per_device_train_batch_size=roberta_params["batch_size"],
    per_device_eval_batch_size=roberta_params["batch_size"],
    num_train_epochs=roberta_params["epochs"],
    weight_decay=roberta_params["weight_decay"],
    load_best_model_at_end=True,
    logging_dir=os.path.join(paths["results"], "logs"),
    seed=seed
)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}

# Integració dels pesos de classe

print("Calculant els pesos de les classes per compensar el desequilibri...")
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_encoded),
    y=y_train_encoded
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        # S'aplica la CrossEntropyLoss passant-li el tensor de pesos mogut al dispositiu correcte (GPU/CPU)
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor.to(model.device))
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss
# ==============================================================================

# Instanciem el nou WeightedTrainer en lloc del Trainer per defecte
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Iniciant l'entrenament amb penalització de classes desequilibrades...")
trainer.train()
print("Model entrenat amb èxit.")

**4. Avaluació i exportació de resultats**

In [ ]:
print("\n--- Avaluació al conjunt de Validació ---")
val_predictions = trainer.predict(val_dataset)
y_val_pred = np.argmax(val_predictions.predictions, axis=1)
y_val_pred_labels = label_encoder.inverse_transform(y_val_pred)
print(classification_report(df_val['label'], y_val_pred_labels))

print("\n--- Avaluació al conjunt de Test ---")
test_predictions = trainer.predict(test_dataset)
y_test_pred = np.argmax(test_predictions.predictions, axis=1)
y_test_pred_labels = label_encoder.inverse_transform(y_test_pred)
print(classification_report(df_test['label'], y_test_pred_labels))

# Guardar les prediccions per a la Late Fusion
df_val_results = df_val.copy()
df_val_results['roberta_pred'] = y_val_pred_labels
# Extraiem també les probabilitats (softmax) per a la fusió tardana
df_val_results['roberta_probs'] = torch.nn.functional.softmax(torch.tensor(val_predictions.predictions), dim=-1).numpy().tolist()

df_test_results = df_test.copy()
df_test_results['roberta_pred'] = y_test_pred_labels
df_test_results['roberta_probs'] = torch.nn.functional.softmax(torch.tensor(test_predictions.predictions), dim=-1).numpy().tolist()

df_val_results.to_pickle(os.path.join(paths["results"], "roberta_val_predictions.pkl"))
df_test_results.to_pickle(os.path.join(paths["results"], "roberta_test_predictions.pkl"))

# Exportar el model i eines
trainer.save_model(os.path.join(paths["models"], "roberta_best_model"))
tokenizer.save_pretrained(os.path.join(paths["models"], "roberta_best_model"))
joblib.dump(label_encoder, os.path.join(paths["models"], "label_encoder.pkl"))

print("Procés finalitzat i arxius guardats.")